# Morocco WC2022: Defensive Efficiency & Attacking Precision
### Quantifying the Blueprint: How Morocco Reached the 2022 World Cup Semifinals
*Hachad Solutions LLC · March 2026 · StatsBomb Open Data*

## Motivation

Morocco became the first African and Arab nation to reach a World Cup  semifinal in 2022, eliminating Belgium, Spain, and Portugal along the way before falling to France. Post-tournament tactical analysis, the FIFA Technical Study Group report, and academic profiling work all highlighted a compact defensive structure, disciplined low-to-mid block, and effective transitions as the hallmarks of their run.

This project takes those observations as a starting point and interrogates them with StatsBomb event and 360 data  to see what the numbers actually say at the event level. All existing analysis used aggregated post-match report data. This project uses StatsBomb's event-level and 360 spatial data — a level of granularity no prior published analysis has applied to this specific question.

See `REFERENCES.md` for the prior work that informed this framing.

## Analytical Questions

This project asks three open questions; the answers will emerge from the data:

**Q1: Defensive structure:**  
What does Morocco's defensive behaviour actually look like at the event level across 7 matches? Where did they win the ball back, and what quality of chance did opponents generate as a result? Which individual players or spatial zones drove that structure?

**Q2: Attacking efficiency:**  
Was Morocco's attacking output a reflection of genuine chance quality, or did they outperform their xG? What do their goals look like spatially compared to their baseline expected threat?

**Q3: Broader applicability and WC2026 relevance:**
Can Morocco's WC2022 performance be characterised as a replicable blueprint rather than a one-off? Using Morocco's quantified defensive and offensive profile as a reference point, which qualified WC2026 nations share a similar competitive tier and could plausibly adopt a similar approach? This phase draws on additional data sources beyond StatsBomb to profile WC2026 participants and map them against Morocco's blueprint.

## Notebook Structure

1. Setup & Data Loading
2. Morocco Matches Overview
3. Defensive Analysis
   - 3a. Pressure & defensive intervention map
   - 3b. Opponent shot quality: location and xG
   - 3c. Individual contributions to defensive structure
4. Offensive Analysis
   - 4a. Morocco shot map and xG
   - 4b. Goals vs. xG baseline: finishing efficiency
5. Tournament-level Comparison (all 32 teams)
6. Findings & Conclusions

## Data Source

**StatsBomb Open Data: FIFA World Cup 2022**  
`competition_id = 43` · `season_id = 106`  
64 matches · 32 teams · Full event + 360 freeze-frame data  
https://github.com/statsbomb/open-data

*Data accessed via `statsbombpy` SDK. Per StatsBomb's user agreement, 
this project credits StatsBomb as the data source for all analysis 
derived from their data.*

---
## 1. Setup & Data Loading

In [21]:
# Data source
from statsbombpy import sb

# Data wrangling
import pandas as pd
import numpy as np

# Database
import sqlite3
from pathlib import Path
import ast

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from mplsoccer import Pitch, VerticalPitch

# FRMF / Morocco national team color palette
MOROCCO_RED   = '#E50011'
MOROCCO_GREEN = '#17A376'
MOROCCO_GOLD  = '#D29D63'
MOROCCO_WHITE = '#FFFFFF'
MOROCCO_DARK  = '#1a0a0a'

# Settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
%matplotlib inline

### 1.1 Load Morocco Matches

In [6]:
# Load all WC2022 matches
matches = sb.matches(competition_id=43, season_id=106)

# Filter to Morocco matches only
morocco_matches = matches[
    (matches['home_team'] == 'Morocco') |
    (matches['away_team'] == 'Morocco')    
].reset_index(drop=True)

# Display Match Overview
morocco_matches[[
    'match_id', 'match_date', 'competition_stage', 'home_team', 'home_score', 'away_team', 'away_score'
]].sort_values('match_date')

,match_id,match_date,competition_stage,home_team,home_score,away_team,away_score
5,3857277,2022-11-23,Group Stage,Morocco,0,Croatia,0
4,3857283,2022-11-27,Group Stage,Belgium,0,Morocco,2
6,3857276,2022-12-01,Group Stage,Canada,1,Morocco,2
3,3869220,2022-12-06,Round of 16,Morocco,0,Spain,0
0,3869486,2022-12-10,Quarter-finals,Morocco,1,Portugal,0
2,3869552,2022-12-14,Semi-finals,France,2,Morocco,0
1,3869684,2022-12-17,3rd Place Final,Croatia,2,Morocco,1


### 1.2 Full Data Ingestion

In [7]:
# Fetch all events across all 7 Morocco matches
match_ids = morocco_matches['match_id'].tolist()

events_list = []
for mid in match_ids:
    print(f"Fetching events for match_id {mid}...")
    df = sb.events(match_id=mid)
    df['match_id'] = mid #to tag each row with its respective match
    events_list.append(df)

events_df = pd.concat(events_list, ignore_index=True)
print(f"\nTotal Events Loaded: {len(events_df):,}") #add commas as thousands separators
print(f"Number of Columns: {events_df.shape[1]}")

Fetching events for match_id 3869486...
Fetching events for match_id 3869684...
Fetching events for match_id 3869552...
Fetching events for match_id 3869220...
Fetching events for match_id 3857283...
Fetching events for match_id 3857277...
Fetching events for match_id 3857276...

Total Events Loaded: 25,971
Number of Columns: 105


In [8]:
# Fetch all 360 freeze-frame data across all 7 Morocco matches

frames_list = []
for mid in match_ids:
    print(f"Fetching 360 freeze frames for match_id: {mid}...")
    df = sb.frames(match_id=mid)
    df['match_id'] = mid
    frames_list.append(df)

frames_df = pd.concat(frames_list, ignore_index=True)
print(f"\nTotal Freeze Frames Loaded: {len(frames_df):,}")
print(f"Number of Columns: {frames_df.shape[1]}")

Fetching 360 freeze frames for match_id: 3869486...
Fetching 360 freeze frames for match_id: 3869684...
Fetching 360 freeze frames for match_id: 3869552...
Fetching 360 freeze frames for match_id: 3869220...
Fetching 360 freeze frames for match_id: 3857283...
Fetching 360 freeze frames for match_id: 3857277...
Fetching 360 freeze frames for match_id: 3857276...

Total Freeze Frames Loaded: 344,302
Number of Columns: 7


In [9]:
# Write to SQLite DB
db_path = Path('../data/processed/morocco.db')
db_path.parent.mkdir(parents=True, exist_ok=True) # ensures that directory structure exists so code doesn't crash when it tries to write the db

con = sqlite3.connect(db_path)

# Serialize any nested columns to JSON strings for SQLite compatibility
for col in events_df.columns:
    if events_df[col].dtype == object:
        events_df[col] = events_df[col].apply(
            lambda x: str(x) if isinstance(x, (list, dict)) else x
        )

for col in frames_df.columns:
    if frames_df[col].dtype == object:
        frames_df[col] = frames_df[col].apply(
            lambda x: str(x) if isinstance(x, (list, dict)) else x
        )

events_df.to_sql('events', con, if_exists='replace', index=False, method='multi', chunksize=100) # method = multi : instead of sending rows one by one (which is slow), it groups them into batches
frames_df.to_sql('frames', con, if_exists='replace', index=False, method='multi', chunksize=100)

# Verify row counts
events_count = pd.read_sql("SELECT COUNT(*) AS number_of_events FROM events", con).iloc[0,0] # iloc[0,0] converts the result from a table containing a number into just a raw integer
frames_count = pd.read_sql("SELECT COUNT(*) AS number_of_frames FROM frames", con).iloc[0,0]

In [10]:
# SQL inspection: event type breakdown
event_types = pd.read_sql("""
    SELECT type, COUNT(*) AS count
    FROM events
    GROUP BY type
    ORDER BY count DESC
    LIMIT 15
""", con)

print("\nTop 15 event types:")
event_types


Top 15 event types:


,type,count
0,Pass,7530
1,Ball Receipt*,6932
2,Carry,6094
3,Pressure,1968
4,Ball Recovery,642
5,Duel,450
6,Clearance,249
7,Block,227
8,Miscontrol,221
9,Dribble,214


--- 
## 2. Morocco Matches Overview

In [12]:
# Clean match summary table
# Identify Morocco's score and oppponent for each match
def get_morocco_summary(row):
    if row['home_team'] == 'Morocco':
        return pd.Series({
            'opponent': row['away_team'],
            'morocco_score': row['home_score'],
            'opponent_score': row['away_score'],
            'result': 'W' if row['home_score'] > row['away_score'] else ('D' if row['home_score'] == row['away_score'] else 'L')
        })
    else:
        return pd.Series({
            'opponent': row['home_team'],
            'morocco_score': row['away_score'],
            'opponent_score': row['home_score'],
            'result': 'W' if row['away_score'] > row['home_score'] else ('D' if row['away_score'] == row['home_score'] else 'L')
        })

summary_cols = morocco_matches[['match_id', 'match_date', 'competition_stage', 'home_team', 'home_score', 'away_team', 'away_score']].sort_values('match_date').copy()

extra = summary_cols.apply(get_morocco_summary, axis = 1) # axis=1 moves along axis 1 (the column axis) meaning it collapses columns and applies the function to each row.

match_summary = pd.concat([summary_cols[['match_id', 'match_date', 'competition_stage']].reset_index(drop=True), extra.reset_index(drop=True)], axis=1) # axis = 1 stacks DataFrames side by side (column-wise) rather than stacking them on top of each other.

match_summary

,match_id,match_date,competition_stage,opponent,morocco_score,opponent_score,result
0,3857277,2022-11-23,Group Stage,Croatia,0,0,D
1,3857283,2022-11-27,Group Stage,Belgium,2,0,W
2,3857276,2022-12-01,Group Stage,Canada,2,1,W
3,3869220,2022-12-06,Round of 16,Spain,0,0,D
4,3869486,2022-12-10,Quarter-finals,Portugal,1,0,W
5,3869552,2022-12-14,Semi-finals,France,0,2,L
6,3869684,2022-12-17,3rd Place Final,Croatia,1,2,L


In [15]:
# Persist match metadata for join capability
match_summary.to_sql('matches', con, if_exists='replace', index=False)

7

In [18]:
# Event counts per match: sanity check and overview

pd.read_sql("""
    SELECT
        e.match_id,
        m.match_date,
        m.competition_stage,
        m.opponent,
        m.morocco_score,
        m.opponent_score,
        m.result,
        COUNT(*) AS total_events,
        SUM(CASE WHEN e.team = 'Morocco' THEN 1 ELSE 0 END) AS morocco_events,
        SUM(CASE WHEN e.team != 'Morocco' THEN 1 ELSE 0 END) AS opponent_events
    FROM events e
    JOIN matches m ON e.match_id = m.match_id
    GROUP BY e.match_id
    ORDER BY m.match_date
""", con, index_col='match_id' )

,match_date,competition_stage,opponent,morocco_score,opponent_score,result,total_events,morocco_events,opponent_events
match_id,,,,,,,,,
3857277,2022-11-23,Group Stage,Croatia,0,0,D,3719,1498,2221
3857283,2022-11-27,Group Stage,Belgium,2,0,W,3621,1361,2260
3857276,2022-12-01,Group Stage,Canada,2,1,W,3389,1472,1917
3869220,2022-12-06,Round of 16,Spain,0,0,D,4750,1371,3379
3869486,2022-12-10,Quarter-finals,Portugal,1,0,W,3381,1111,2270
3869552,2022-12-14,Semi-finals,France,0,2,L,3551,2071,1480
3869684,2022-12-17,3rd Place Final,Croatia,1,2,L,3560,1798,1762


---
## 3. Defensive Analysis
### 3a. Pressure & Defensive Intervention Map

In [19]:
# Before anything, first inspect available column in events table (although already available in sb documentation, good to have directly here inline)
columns = pd.read_sql("SELECT * FROM events LIMIT 1", con).columns.tolist()
print('\n'.join(columns))

ball_receipt_outcome
ball_recovery_recovery_failure
block_deflection
carry_end_location
clearance_aerial_won
clearance_body_part
clearance_head
clearance_left_foot
clearance_other
clearance_right_foot
counterpress
dribble_nutmeg
dribble_outcome
dribble_overrun
duel_outcome
duel_type
duration
foul_committed_advantage
foul_committed_card
foul_committed_offensive
foul_committed_type
foul_won_advantage
foul_won_defensive
goalkeeper_body_part
goalkeeper_end_location
goalkeeper_outcome
goalkeeper_position
goalkeeper_punched_out
goalkeeper_technique
goalkeeper_type
id
index
injury_stoppage_in_chain
interception_outcome
location
match_id
minute
miscontrol_aerial_won
off_camera
out
pass_aerial_won
pass_angle
pass_assisted_shot_id
pass_body_part
pass_cross
pass_cut_back
pass_deflected
pass_end_location
pass_goal_assist
pass_height
pass_inswinging
pass_length
pass_outcome
pass_outswinging
pass_recipient
pass_recipient_id
pass_shot_assist
pass_switch
pass_technique
pass_through_ball
pass_type
peri

In [25]:
# Helper function: parse serialized location string back to x,y coordinates
def parse_coords(loc):
    try:
        coords = ast.literal_eval(loc) # parses a string that looks like a Python literal ("[34.2, 45.6]") and evaluates it back into the actual Python object ([34.2, 45.6])
        return coords[0], coords[1]
    except:
        return None, None
    
# Pull Morocco's defensive actions from DB with match context
morocco_defensive = pd.read_sql("""
    SELECT
        e.id,
        e.match_id,
        e.type,
        e.player,
        e.position,
        e.location,
        e.minute,
        e.period,
        e.duration,
        e.under_pressure,
        e.counterpress,
        e.play_pattern,
        e.interception_outcome,
        e.block_save_block,
        e.block_offensive,
        e.clearance_aerial_won,
        e.clearance_body_part,
        e.ball_recovery_offensive,
        e.duel_type,
        e.duel_outcome,
        m.opponent,
        m.competition_stage,
        m.result
    FROM events e
    JOIN matches m ON e.match_id = m.match_id
    WHERE e.team = 'Morocco'
      AND (
          e.type IN ('Pressure', 'Interception', 'Block', 'Clearance', 'Ball Recovery')
          OR (e.type = 'Duel' AND e.duel_type = 'Tackle')
      )
""", con)

# Parse x, y from location string
morocco_defensive[['x', 'y']] = morocco_defensive['location'].apply(
    lambda loc: pd.Series(parse_coords(loc))
)

print(f"Morocco defensive actions: {len(morocco_defensive):,}")
print(f"\nBy type:")
print(morocco_defensive['type'].value_counts().to_string())
print(f"\nBy match:")
print(morocco_defensive.groupby('match_id')['type'].count()
      .reset_index()
      .rename(columns={'type': 'defensive_actions'})
      .merge(match_summary[['match_id', 'match_date', 'opponent', 'competition_stage', 'result']], on ='match_id')
      .sort_values('match_date')
      .to_string(index=False))

Morocco defensive actions: 1,945

By type:
Pressure         1085
Ball Recovery     329
Clearance         171
Duel              137
Block             134
Interception       89

By match:
 match_id  defensive_actions match_date opponent competition_stage result
  3857277                306 2022-11-23  Croatia       Group Stage      D
  3857283                272 2022-11-27  Belgium       Group Stage      W
  3857276                278 2022-12-01   Canada       Group Stage      W
  3869220                328 2022-12-06    Spain       Round of 16      D
  3869486                300 2022-12-10 Portugal    Quarter-finals      W
  3869552                216 2022-12-14   France       Semi-finals      L
  3869684                245 2022-12-17  Croatia   3rd Place Final      L
